In [0]:
%sql
CREATE CATALOG IF NOT EXISTS hedis;
USE CATALOG hedis;
CREATE SCHEMA IF NOT EXISTS bronze;
USE SCHEMA bronze;


In [0]:
# ============================================================
# BRONZE LAYER — Secure ingestion via Unity Catalog External Location
# No hardcoded keys. No spark.conf credential settings.
# Access is governed by IAM Role via Storage Credential.
# ============================================================

from pyspark.sql import functions as F

# The External Location URL you created in Step 5
EXTERNAL_LOCATION = "s3://hedis-analytics-jyothi-2026/raw/v2/"

tables = [
    "allergies", "careplans", "conditions", "devices",
    "encounters", "imaging_studies", "immunizations", "medications", "observations", 
    "organizations", "patients", "payer_transitions", "payers", "procedures",  "providers", "supplies"
]

for table in tables:
    print(f"Reading: {table}")
    df = spark.read.csv(
        f"{EXTERNAL_LOCATION}{table}.csv",
        header=True,
        inferSchema=True,
        timestampFormat="yyyy-MM-dd'T'HH:mm'Z'"
    ).withColumn("_ingested_at", F.current_timestamp()) \
     .withColumn("_source_file", F.lit(f"{table}.csv"))

    # Write as managed Delta table in Unity Catalog
    df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"hedis.bronze.{table}")
    print(f"  ✓ {df.count()} rows → hedis.bronze.{table}")

print("\n✅ Bronze ingestion complete — all tables in Unity Catalog")


Reading: allergies
  ✓ 20881 rows → hedis.bronze.allergies
Reading: careplans
  ✓ 74378 rows → hedis.bronze.careplans
Reading: conditions
  ✓ 819638 rows → hedis.bronze.conditions
Reading: devices
  ✓ 128714 rows → hedis.bronze.devices
Reading: encounters
  ✓ 1338608 rows → hedis.bronze.encounters
Reading: imaging_studies
  ✓ 2226501 rows → hedis.bronze.imaging_studies
Reading: immunizations
  ✓ 334643 rows → hedis.bronze.immunizations
Reading: medications
  ✓ 1122801 rows → hedis.bronze.medications
Reading: observations
  ✓ 17137977 rows → hedis.bronze.observations
Reading: organizations
  ✓ 12711 rows → hedis.bronze.organizations
Reading: patients
  ✓ 22887 rows → hedis.bronze.patients
Reading: payer_transitions
  ✓ 840179 rows → hedis.bronze.payer_transitions
Reading: payers
  ✓ 40 rows → hedis.bronze.payers
Reading: procedures
  ✓ 3667575 rows → hedis.bronze.procedures
Reading: providers
  ✓ 12711 rows → hedis.bronze.providers
Reading: supplies
  ✓ 586936 rows → hedis.bronze.suppli